In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/appointments_clean.csv")
df["reminder_sent"] = df["reminder_sent"].map({"Yes": 1, "No": 0})
X = df[["age", "distance_km", "fee_inr", "lead_days",
        "prior_no_shows", "reminder_sent", "booking_channel",
        "visit_type"]]
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))
print("No-shows in test set:", y_test.sum())
baseline_accuracy = (y_test == 0).mean()
print("Baseline accuracy:", round(baseline_accuracy, 3))
print("Baseline accuracy percentage:", round(baseline_accuracy * 100, 1), "%")

Training set size: 792
Testing set size: 198
No-shows in test set: 49
Baseline accuracy: 0.753
Baseline accuracy percentage: 75.3 %


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# Scale the features using the training data
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train the logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nPrecision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 Score:", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

print("\nFeature Coefficients:")
for name, coef in zip(X.columns, model.coef_[0]):
    print(name, ":", round(coef, 4))


# Calculate the odds ratio for reminder_sent
idx = list(X.columns).index("reminder_sent")

# Odds ratio using the scaled coefficient
scaled_or = np.exp(model.coef_[0][idx])

# Adjust the coefficient to represent a change from 0 to 1
actual_coef = model.coef_[0][idx] / scaler.scale_[idx]
actual_or = np.exp(actual_coef)

print("\nReminder Sent Analysis:")
print("Odds ratio using scaled coefficient:", round(scaled_or, 4))
print("Odds ratio for No to Yes:", round(actual_or, 4))

Accuracy: 0.788

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.63
Recall: 0.347
F1 Score: 0.447
ROC-AUC: 0.789

Feature Coefficients:
age : -0.2539
distance_km : 0.2534
fee_inr : 0.4528
lead_days : 0.6145
prior_no_shows : 0.6496
reminder_sent : -0.5807
booking_channel_App : -0.0468
booking_channel_Phone : 0.2159
booking_channel_Walk-In : -0.1864
visit_type_Consultation : 0.0157
visit_type_Follow-up : -0.0157

Reminder Sent Analysis:
Odds ratio using scaled coefficient: 0.5595
Odds ratio for No to Yes: 0.3094


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Decision tree without any depth limit
tree1 = DecisionTreeClassifier(random_state=42)
tree1.fit(X_train, y_train)

# Calculate training and testing accuracy
train_acc1 = accuracy_score(y_train, tree1.predict(X_train))
test_acc1 = accuracy_score(y_test, tree1.predict(X_test))

print("Decision Tree without depth limit")
print("Training Accuracy:", round(train_acc1, 3))
print("Testing Accuracy:", round(test_acc1, 3))


# Decision tree with maximum depth of 3
tree2 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree2.fit(X_train, y_train)

# Calculate training and testing accuracy
train_acc2 = accuracy_score(y_train, tree2.predict(X_train))
test_acc2 = accuracy_score(y_test, tree2.predict(X_test))

print("\nDecision Tree with max_depth=3")
print("Training Accuracy:", round(train_acc2, 3))
print("Testing Accuracy:", round(test_acc2, 3))

Decision Tree without depth limit
Training Accuracy: 1.0
Testing Accuracy: 0.672

Decision Tree with max_depth=3
Training Accuracy: 0.822
Testing Accuracy: 0.778


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
import pandas as pd

# Train the random forest model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Make predictions on the test data
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# Evaluate the random forest
rf_acc = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest Results")
print("Accuracy:", round(rf_acc, 3))
print("Recall:", round(rf_recall, 3))
print("ROC-AUC:", round(rf_auc, 3))


# Find the three most important features
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\nTop 3 Important Features:")
print(importance.head(3))


# Compare all three models
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree (Depth 3)", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, model.predict(X_test_s)),
        accuracy_score(y_test, tree2.predict(X_test)),
        rf_acc
    ],
    "Recall": [
        recall_score(y_test, model.predict(X_test_s)),
        recall_score(y_test, tree2.predict(X_test)),
        rf_recall
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1]),
        roc_auc_score(y_test, tree2.predict_proba(X_test)[:, 1]),
        rf_auc
    ]
})

print("\nModel Comparison:")
print(results.round(3).to_string(index=False))

Random Forest Results
Accuracy: 0.747
Recall: 0.306
ROC-AUC: 0.706

Top 3 Important Features:
lead_days      0.219905
age            0.196788
distance_km    0.194739
dtype: float64

Model Comparison:
                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression     0.788   0.347    0.789
Decision Tree (Depth 3)     0.778   0.224    0.700
          Random Forest     0.747   0.306    0.706


In [6]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Try different settings for the random forest
params = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    params,
    cv=5,
    scoring="roc_auc"
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Cross-Validation ROC-AUC:", round(grid.best_score_, 3))

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best Cross-Validation ROC-AUC: 0.767


In [7]:
# Create a pipeline for logistic regression
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Create the other models
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Calculate cross-validation ROC-AUC for each model
lr_auc = cross_val_score(
    lr_pipe, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tree_auc = cross_val_score(
    tree_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

rf_auc = cross_val_score(
    rf_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

# Compare all four models, including the tuned random forest
scores = {
    "Logistic Regression": lr_auc,
    "Decision Tree (Depth 3)": tree_auc,
    "Random Forest": rf_auc,
    "Tuned Random Forest": grid.best_score_
}

print("\nCross-validation ROC-AUC:")
for name, score in scores.items():
    print(name, ":", round(score, 3))

# Select the model with the highest cross-validation ROC-AUC
best_name = max(scores, key=scores.get)

if best_name == "Logistic Regression":
    best_model = lr_pipe
elif best_name == "Decision Tree (Depth 3)":
    best_model = tree_model
elif best_name == "Random Forest":
    best_model = rf_model
else:
    best_model = grid.best_estimator_

print("\nSelected Model:", best_name)
print("Selected CV ROC-AUC:", round(scores[best_name], 3))


Cross-validation ROC-AUC:
Logistic Regression : 0.774
Decision Tree (Depth 3) : 0.678
Random Forest : 0.747
Tuned Random Forest : 0.767

Selected Model: Logistic Regression
Selected CV ROC-AUC: 0.774


In [8]:
# Get out-of-fold probabilities using only the training data
oof = cross_val_predict(
    best_model, X_train, y_train, cv=5, method="predict_proba"
)[:, 1]

# Compare the two thresholds
for threshold in [0.5, 0.35]:
    pred = (oof >= threshold).astype(int)

    precision = precision_score(y_train, pred, zero_division=0)
    recall = recall_score(y_train, pred, zero_division=0)
    f1 = f1_score(y_train, pred, zero_division=0)
    flagged = pred.sum()

    print("\nThreshold:", threshold)
    print("Precision:", round(precision, 3))
    print("Recall:", round(recall, 3))
    print("F1 Score:", round(f1, 3))
    print("Number flagged:", flagged)

# Select the threshold with the higher F1 score
f1_05 = f1_score(y_train, (oof >= 0.5).astype(int), zero_division=0)
f1_035 = f1_score(y_train, (oof >= 0.35).astype(int), zero_division=0)

threshold = 0.5 if f1_05 >= f1_035 else 0.35

print("\nSelected Threshold:", threshold)


Threshold: 0.5
Precision: 0.575
Recall: 0.249
F1 Score: 0.347
Number flagged: 73

Threshold: 0.35
Precision: 0.513
Recall: 0.467
F1 Score: 0.489
Number flagged: 154

Selected Threshold: 0.35


In [9]:
# Train the selected model using the complete training set
best_model.fit(X_train, y_train)

# Get no-show probabilities for the test set
probs = best_model.predict_proba(X_test)[:, 1]

# Classify appointments using the selected threshold
test_pred = (probs >= threshold).astype(int)

# Evaluate the final model
print("\nFinal Test Results")
print("Selected Model:", best_name)
print("Selected Threshold:", threshold)

print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 3))
print("Number of appointments flagged:", test_pred.sum())
print("ROC-AUC:", round(roc_auc_score(y_test, probs), 3))


Final Test Results
Selected Model: Logistic Regression
Selected Threshold: 0.35
Precision: 0.571
Recall: 0.571
Number of appointments flagged: 49
ROC-AUC: 0.789


In [10]:
# Create a file containing the risk scores for each test appointment
risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": test_pred
})

# Save the risk scores
risk_scores.to_csv("test_risk_scores.csv", index=False)

print("\nTest risk scores saved successfully!")
print(risk_scores.head())


Test risk scores saved successfully!
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0
